In [ ]:
# Imports
import sys
import pandas as pd

import os
import json 
from pathlib import Path

# Access packages from the root
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")))

from app.services.quiz_router import handle_quiz_request
from app.agent.rag.ingestion.data_ingestion import process_and_load_file
from app.agent.rag.ingestion.embeddings import get_embedding_model, generate_embeddings
from app.agent.rag.ingestion.vector_store import get_vector_store, add_documents_to_chroma
from app.agent.rag.chains.notes_chain import run_notes_chain
from app.agent.rag.retrieval.retriever import get_semantic_chunks, get_topic_chunks
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from typing import Dict, List
from langchain_google_genai import ChatGoogleGenerativeAI
from app.agent.rag.chains.eval_chain import run_eval_chain


In [2]:

load_dotenv()
api_key = os.environ.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",  # The current efficient workhorse model
    project="denseless",       # Replace with your actual Project ID
    location="us-central1",              # Or your preferred region
    vertexai=True,
)

json_eval_dataset = {}
quizzes: Dict[str, List] = {}

INFO:google_genai._api_client:The user provided project/location will take precedence over the Vertex AI API key from the environment variable.


In [ ]:

topics = ['ADDER CIRCUIT.pdf', 'Disaster Recovery Plan.pdf', 'Incident Response.pdf', 'PROCESS MANAGEMENT.pdf', 'Software Engineering Intro.pdf']

# Add all topics to store first - Added
# for topic in topics:
#     topic_name = topic.removesuffix('.pdf')
#     docs = process_and_load_file(f"../pdfs/{topic}")
#     embedder = get_embedding_model()
#     vectors = generate_embeddings(docs, embedder)
#     store = get_vector_store(student_id="1019", embedder=embedder)
#     add_documents_to_chroma(store, vectors, docs, False, "FQ", topic_name, topic_name)


embedder = get_embedding_model()
store = get_vector_store(student_id="1019", embedder=embedder)
embedder = get_embedding_model()

# Generate 10 quizzes for each topic
for topic in topics:
    topic_name = topic.removesuffix('.pdf')
    quizzes[topic] = []

    for _ in range(10):
        result = handle_quiz_request(
            student_id      = "student_1019",
            course          = "FQ",
            topic           = topic_name,
            store           = store,
            llm             = llm,
            simulated_today = "2026-05-11"
        )
        print(result["status"])
        quiz_file_name = result["saved_path"].removeprefix("data/quizzes/")
        quizzes[topic].append(quiz_file_name)


In [7]:
os.listdir("../../data/quizzes")

['student_1019_adder_circuit_20260525_121048.json',
 'student_1019_adder_circuit_20260525_121056.json',
 'student_1019_adder_circuit_20260525_121108.json',
 'student_1019_adder_circuit_20260525_121115.json',
 'student_1019_adder_circuit_20260525_121122.json',
 'student_1019_adder_circuit_20260525_121129.json',
 'student_1019_adder_circuit_20260525_121137.json',
 'student_1019_adder_circuit_20260525_121143.json',
 'student_1019_adder_circuit_20260525_121201.json',
 'student_1019_adder_circuit_20260525_121208.json',
 'student_1019_disaster_recovery_plan_20260525_121220.json',
 'student_1019_disaster_recovery_plan_20260525_121228.json',
 'student_1019_disaster_recovery_plan_20260525_121236.json',
 'student_1019_disaster_recovery_plan_20260525_121246.json',
 'student_1019_disaster_recovery_plan_20260525_121252.json',
 'student_1019_disaster_recovery_plan_20260525_121259.json',
 'student_1019_disaster_recovery_plan_20260525_121305.json',
 'student_1019_disaster_recovery_plan_20260525_121310

In [ ]:
# Offline
topics = ['ADDER CIRCUIT.pdf', 'Disaster Recovery Plan.pdf', 'Incident Response.pdf', 'PROCESS MANAGEMENT.pdf', 'Software Engineering Intro.pdf']
quiz_jsons = os.listdir("../../data/quizzes")

i = 0
for topic in topics: 
    topic_name = topic.removesuffix('.pdf')
    quizzes[topic] = []

    for _ in range(10):
        # print(i)
        quiz_file_name = quiz_jsons[i]
        quizzes[topic].append(quiz_file_name)
        i+=1

print(quizzes)

{'ADDER CIRCUIT.pdf': ['student_1019_adder_circuit_20260525_121048.json', 'student_1019_adder_circuit_20260525_121056.json', 'student_1019_adder_circuit_20260525_121108.json', 'student_1019_adder_circuit_20260525_121115.json', 'student_1019_adder_circuit_20260525_121122.json', 'student_1019_adder_circuit_20260525_121129.json', 'student_1019_adder_circuit_20260525_121137.json', 'student_1019_adder_circuit_20260525_121143.json', 'student_1019_adder_circuit_20260525_121201.json', 'student_1019_adder_circuit_20260525_121208.json'], 'Disaster Recovery Plan.pdf': ['student_1019_disaster_recovery_plan_20260525_121220.json', 'student_1019_disaster_recovery_plan_20260525_121228.json', 'student_1019_disaster_recovery_plan_20260525_121236.json', 'student_1019_disaster_recovery_plan_20260525_121246.json', 'student_1019_disaster_recovery_plan_20260525_121252.json', 'student_1019_disaster_recovery_plan_20260525_121259.json', 'student_1019_disaster_recovery_plan_20260525_121305.json', 'student_1019_d

In [ ]:
# Answer all quizzes manually - Done with Claude and GPT Chat UI

In [18]:
# Evaluate each quiz and store score and feedback
i = 1
for topic in topics:
    topic_name = topic.removesuffix('.pdf')
     
    for quiz_file_name in quizzes[topic]:
        quiz_file_name = os.path.basename(quiz_file_name)
        response = run_eval_chain(
            student_id     = "student_1019",
            topic          = topic_name,
            quiz_phase     = "revision",
            quiz_path      = f"../../data/quizzes/{quiz_file_name}",
            llm            = llm,
            simulated_date = "2026-05-11"
        )
        feedback = response.content["feedback"]
        score = response.content["questions"][0]["score"] * 100

        key = str(i).zfill(3)
        json_eval_dataset[key] = {
            "topic_title"      : topic_name,
            "quiz_file"        : quiz_file_name,
            "overall_quiz_score": score,
            "feedback_given"   : feedback,
        }
        i += 1


  [token_service] Tokens remaining for 'student_1019': 7,144,326
  [token_guard] Checking tokens — student: student_1019 | remaining: 7144326 | chain: run_eval_chain
[eval_chain] Quiz loaded from ..\..\data\quizzes\student_1019_adder_circuit_20260525_121048.json
[eval_chain] Starting eval — student: 'student_1019' | topic: 'ADDER CIRCUIT' | phase: 'revision' | questions: 10
[eval_chain] Grading question 1/10...


INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[eval_chain] Question graded. Score: 1.0
[eval_chain] Grading question 2/10...


INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[eval_chain] Question graded. Score: 1.0
[eval_chain] Grading question 3/10...


...

[eval_chain] Question graded. Score: 0.0
[eval_chain] Grading question 10/10...


INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[eval_chain] Question graded. Score: 1.0
[eval_chain] All 10 questions graded.
[eval_chain] Total score: 7.15 / 10
[eval_chain] Pre-resolution — strong: {'The Term Software', 'Today’s software is comprised of', 'The Term Software Engineering', 'Course Contents', 'The Term Engineering'} | weak: {'Computer Science VS Software Engineering', 'Today’s software is comprised of'}
[eval_chain] Post-resolution — strong: {'The Term Software Engineering', 'Course Contents', 'The Term Software', 'The Term Engineering'} | weak: {'Computer Science VS Software Engineering', 'Today’s software is comprised of'}


INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[eval_chain] Session feedback generated.
[eval_chain] Section classifications updated for topic 'Software Engineering Intro'.
[eval_chain] Revision score → retention (attempt 5).
[eval_chain] Revision date entry for 2026-05-11 marked completed.
[eval_chain] Graded quiz saved to ..\..\data\quizzes\student_1019_software_engineering_intro_20260525_122010.json
[eval_chain] Eval chain complete for student 'student_1019'.
  [token_service] Deducted 6,010 tokens — student: 'student_1019' | chain: run_eval_chain | in: 5,372 | out: 638 | remaining: 6,839,047
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 6010 | remaining: 6839047


In [19]:
# Export to a json
output_path = Path(os.getcwd()) / "rag" / "eval_data" / "fq_eval_data.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(json_eval_dataset, f, indent=4, ensure_ascii=False)

print(f"Successfully completed. Exported to {output_path}")

Successfully completed. Exported to c:\Users\USER\Documents\AI projects\DenseLess\app\agent\rag\eval_data\fq_eval_data.json
